# 👥 Notebook 02 — Collaborative filtering og ALS

**Metadata ser filmene. Men hva om vi ser på *menneskene* i stedet?**

Tusenvis av brukere har allerede fortalt oss hva de liker — gjennom handlingene sine.
Nå lar vi det kollektive mønsteret hjelpe Lea. Vi går fra en enkel item-item-metode
til den klassiske arbeidshesten **ALS matrisfaktorisering**.

## Fra nabolag til latente faktorer

### Item-item collaborative filtering

I notebook 01 sammenlignet vi filmer basert på sjangeretiketter. Nå snur vi
perspektivet: i stedet for å lese filmomslaget spør vi *hvem andre så denne filmen,
og hva så de etterpå?*

Teknisk gjør vi dette ved å transponere interaksjonsmatrisen: rader blir filmer,
kolonner blir brukere. Cosine similarity mellom to filmer måler da hvor mye
de deler publikum — uavhengig av sjanger. To filmer kan være like fordi
de *samme menneskene* liker dem, selv om én er en thriller og den andre er drama.

### Matrix factorization

I stedet for å sammenligne rå item-vektorer direkte, lærer vi latente faktorer:

$$R \approx U \cdot V^T$$

Her er $U$ brukerfaktorer og $V$ itemfaktorer. ALS er en effektiv måte å lære dette
på for store, sparsomme implisitte datasett. Mer om dette etter Oppgave 2.

## Oppsett

In [5]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from src.data import load_interactions, load_item_metadata, GENRE_COLS
from src.split import leave_one_out_split, build_sparse_matrix
from src.metrics import recall_at_k, ndcg_at_k, map_at_k
from implicit.als import AlternatingLeastSquares

interactions = load_interactions()
items = load_item_metadata()
train_df, test_df = leave_one_out_split(interactions)
n_users = interactions.user_id.max() + 1
n_items = interactions.item_id.max() + 1
train_matrix = build_sparse_matrix(train_df, n_users, n_items)
user_ids = test_df['user_id'].values
test_items = test_df['item_id'].values
K = 10
LEA_ID = 451

print(f'Matrise: {train_matrix.shape}, nnz={train_matrix.nnz:,}')

Matrise: (15000, 12169), nnz=3,343,357


## 🏋️ Oppgave 2 — Inspiser item-likhet

La oss se hvordan co-view-basert likhet fungerer. Først noen eksempler:

In [6]:
item_sim = cosine_similarity(train_matrix.T, dense_output=True)
np.fill_diagonal(item_sim, 0.0)
sample_items = items.sample(3, random_state=42)

for _, row in sample_items.iterrows():
    item_id = row['item_id']
    if item_id >= item_sim.shape[0]:
        continue
    neighbors = np.argsort(-item_sim[item_id])[:5]
    print(f'\n"{row["title"]}" ligner på:')
    for neighbor_id in neighbors:
        neighbor_row = items[items.item_id == neighbor_id]
        if len(neighbor_row) > 0:
            print(f'  {item_sim[item_id, neighbor_id]:.3f}  {neighbor_row.iloc[0]["title"]}')


"Play It Again, Sam (1972)" ligner på:
  0.404  Hannah and Her Sisters (1986)
  0.374  Broadway Danny Rose (1984)
  0.374  Zelig (1983)
  0.350  Love and Death (1975)
  0.321  Front, The (1976)

"Irma la Douce (1963)" ligner på:
  0.260  Life of Emile Zola, The (1937)
  0.254  Shampoo (1975)
  0.252  Fortune Cookie, The (1966)
  0.251  Barefoot in the Park (1967)
  0.248  Tom Jones (1963)

"After the Thin Man (1936)" ligner på:
  0.669  Another Thin Man (1939)
  0.594  Shadow of the Thin Man (1941)
  0.584  Thin Man Goes Home, The (1945)
  0.554  Song of the Thin Man (1947)
  0.394  Pat and Mike (1952)


### 🏋️ Oppgave 2b — Velg din egen film

Finn en film du kjenner i katalogen og se hvem naboene er.
Gir resultatet mening? Hva fanger likheten som sjanger alene ikke ser?

In [7]:
# Fyll inn en tittel du kjenner (eller del av tittelen)
SEARCH_TITLE = 'Matrix'  # <-- endre dette

matches = items[items['title'].str.contains(SEARCH_TITLE, case=False, na=False)]
if len(matches) == 0:
    print(f'Fant ingen filmer med "{SEARCH_TITLE}" i tittelen.')
else:
    chosen = matches.iloc[0]
    chosen_id = chosen['item_id']
    neighbors = np.argsort(-item_sim[chosen_id])[:5]
    print(f'Naboer til "{chosen["title"]}":')
    for neighbor_id in neighbors:
        nb_row = items[items.item_id == neighbor_id]
        if len(nb_row) > 0:
            print(f'  {item_sim[chosen_id, neighbor_id]:.3f}  {nb_row.iloc[0]["title"]}')

Naboer til "Matrix, The (1999)":
  0.749  Fight Club (1999)
  0.746  Lord of the Rings: The Fellowship of the Ring, The (2001)
  0.722  Lord of the Rings: The Two Towers, The (2002)
  0.716  Star Wars: Episode V - The Empire Strikes Back (1980)
  0.715  Lord of the Rings: The Return of the King, The (2003)


> 💬 **Diskuter**
>
> 1. Ser naboskapet meningsfullt ut?
> 2. Hva fanger item-item-likheten som metadata alene ikke så godt viser?
> 3. Hvorfor er dette fortsatt begrenset?

## 🏋️ Oppgave 3 — Head-to-head: item-item vs ALS

### Hva er ALS, og hvorfor fungerer det?

Tenk deg at hver bruker har en skjult smaksprofil med ~64 dimensjoner —
kanskje «liker spenning», «foretrekker visuelt vakre filmer», «liker komplekse plott».
Hver film har en tilsvarende profil. **ALS prøver å gjette disse profilene**
slik at brukerprofil · filmprofil ≈ hvor mye brukeren likte filmen.

Vi kaller dem *latente* faktorer fordi vi ikke velger dem — de læres
automatisk fra data. De trenger ikke tilsvare sjangre; de kan fange
subtile mønstre som «stemning» eller «regi-stil».

**ALS (Alternating Least Squares)** veksler mellom å oppdatere brukerprofiler
og filmprofiler — litt som å løse et puslespill fra to sider samtidig.
Det gjør den i 15 runder, og når det er ferdig har vi en komprimert
representasjon som fanger sammenhenger item-item ikke ser.

La oss se om det gjør en forskjell. Se spesielt på Recall og NDCG —
tar ALS et tydelig sprang?

> *NB: Også her bygger vi funksjoner manuelt for innsikt.
> I notebook 03–04 bruker vi klassene `ALSRecommender` og `ItemItemRecommender` fra `src/recommenders/`.*

In [8]:
def recommend_item_item(item_sim, train_matrix, user_ids, k=10):
    # Compute all user scores in one vectorized BLAS call:
    # sparse (n_users × n_items) @ dense (n_items × n_items) → (n_users × n_items)
    all_scores = train_matrix[user_ids].dot(item_sim)  # shape: (n_users, n_items)
    recommendations = np.zeros((len(user_ids), k), dtype=np.int32)
    for row_index, user_id in enumerate(user_ids):
        scores = all_scores[row_index].A1 if hasattr(all_scores[row_index], 'A1') else all_scores[row_index]
        scores[train_matrix[user_id].indices] = -np.inf
        recommendations[row_index] = np.argsort(-scores)[:k]
    return recommendations

recs_item_item = recommend_item_item(item_sim, train_matrix, user_ids, k=K)

In [9]:
als = AlternatingLeastSquares(factors=64, regularization=0.01, iterations=15, random_state=42, use_gpu=False)
als.fit(train_matrix, show_progress=True)
recs_als = als.recommend(user_ids, train_matrix[user_ids], N=K, filter_already_liked_items=True)[0]

100%|██████████| 15/15 [00:05<00:00,  2.65it/s]


In [10]:
for name, recs in [('Item-item', recs_item_item), ('ALS', recs_als)]:
    recall_value = recall_at_k(recs, test_items, K)
    ndcg_value = ndcg_at_k(recs, test_items, K)
    map_value = map_at_k(recs, test_items, K)
    print(f'{name:<10} Recall@{K}={recall_value:.4f}  NDCG@{K}={ndcg_value:.4f}  MAP@{K}={map_value:.4f}')

Item-item  Recall@10=0.0542  NDCG@10=0.0284  MAP@10=0.0207
ALS        Recall@10=0.0724  NDCG@10=0.0384  MAP@10=0.0282


In [11]:
LEA_ID = 451
JONAS_ID = 102
lea_idx = np.where(user_ids == LEA_ID)[0]
jonas_idx = np.where(user_ids == JONAS_ID)[0]

for uid, name, idx in [(LEA_ID, 'Lea', lea_idx), (JONAS_ID, 'Jonas', jonas_idx)]:
    if len(idx) > 0:
        for model_name, recs in [('Item-item', recs_item_item), ('ALS', recs_als)]:
            rec_items = recs[idx[0]]
            titles = items.set_index('item_id').loc[rec_items, 'title'].values
            print(f'\n{name} — {model_name}:')
            for rank, title in enumerate(titles, 1):
                print(f'  {rank:>2}. {title}')


Lea — Item-item:
   1. Terminator, The (1984)
   2. Alien (1979)
   3. Raiders of the Lost Ark (Indiana Jones and the Raiders of the Lost Ark) (1981)
   4. Jaws (1975)
   5. Apocalypse Now (1979)
   6. Lethal Weapon (1987)
   7. Die Hard (1988)
   8. Indiana Jones and the Last Crusade (1989)
   9. Indiana Jones and the Temple of Doom (1984)
  10. Back to the Future (1985)

Lea — ALS:
   1. Butch Cassidy and the Sundance Kid (1969)
   2. Cool Hand Luke (1967)
   3. Full Metal Jacket (1987)
   4. Apocalypse Now (1979)
   5. For a Few Dollars More (Per qualche dollaro in più) (1965)
   6. Hunt for Red October, The (1990)
   7. Shawshank Redemption, The (1994)
   8. Tombstone (1993)
   9. Magnificent Seven, The (1960)
  10. African Queen, The (1951)

Jonas — Item-item:
   1. Iron Man 2 (2010)
   2. Iron Man (2008)
   3. Captain America: The First Avenger (2011)
   4. Matrix Reloaded, The (2003)
   5. X-Men: The Last Stand (2006)
   6. Batman Begins (2005)
   7. Matrix Revolutions, The (20

> *Marte ser på tallene:* «Recall gikk opp — men ser Lea **faktisk** bedre filmer?
> Og hva med Jonas — forandret det noe for ham?»

ALS tar et tydelig sprang. Se på listene — ser du forskjellen?

> 💬 **Diskuter**
>
> 1. Hvorfor vinner ALS ofte over item-item?
> 2. Hva er forskjellen på en nabolagsmetode og latente faktorer?
> 3. Ser ALS mer ut som et realistisk produksjonsvalg enn item-item? Hvorfor?

### 🔄 Refleksjon — sjekk prediksjonen din

Gå tilbake til gjetningene du skrev ned i notebook 00.

> 1. Hadde du rett om hva Lea ville like?
> 2. Stemte gjetningen din om popularitetslisten?
> 3. Hva har overrasket deg mest så langt?

---

> *Marte:* «Bra. Nå ser det ut som Lea kan få bedre anbefalinger.
> Men fungerer dette for *alle* — eller bare for mainstream-brukerne?
> Og hva skjer når vi må blande signaler og ta fairness på alvor?
> Amira spør allerede.»

**Neste steg** → `03_hybrid_systems.ipynb`

## Valgfri appendix — faktorsveip og tolkning

Denne delen er nyttig hvis dere vil grave dypere i hvordan ALS oppfører seg,
men den er ikke nødvendig for kjerneflyten.

In [ ]:
import matplotlib.pyplot as plt

factor_values = [16, 32, 64, 128, 256]
recall_per_factor = []
for factor_count in factor_values:
    model = AlternatingLeastSquares(factors=factor_count, regularization=0.01, iterations=15, random_state=42, use_gpu=False)
    model.fit(train_matrix, show_progress=False)
    recs = model.recommend(user_ids, train_matrix[user_ids], N=K, filter_already_liked_items=True)[0]
    recall_per_factor.append(recall_at_k(recs, test_items, K))

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(factor_values, recall_per_factor, 'o-', linewidth=2)
ax.set_xlabel('Antall faktorer')
ax.set_ylabel(f'Recall@{K}')
ax.set_title('ALS: Recall vs antall faktorer')
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity as cosine_sim

user_emb = als.user_factors
lea_emb = user_emb[LEA_ID]
sims = cosine_sim(lea_emb.reshape(1, -1), user_emb)[0]
sims[LEA_ID] = -1
top_neighbors = np.argsort(-sims)[:5]

print('Leas nærmeste naboer i ALS-rommet:')
for neighbor_user_id in top_neighbors:
    neighbor_items = interactions[interactions.user_id == neighbor_user_id].merge(items, on='item_id')
    top_genres = neighbor_items[GENRE_COLS].sum().nlargest(3).index.tolist()
    genre_label = ', '.join(top_genres)
    print(f'  Bruker {neighbor_user_id} (sim={sims[neighbor_user_id]:.3f}, topp-sjangre: {genre_label})')

Leas nærmeste naboer i ALS-rommet:
  Bruker 6282 (sim=0.898, topp-sjangre: Drama, War, Action)
  Bruker 2469 (sim=0.748, topp-sjangre: Drama, War, Action)
  Bruker 6458 (sim=0.727, topp-sjangre: Drama, War, Action)
  Bruker 13068 (sim=0.707, topp-sjangre: Drama, Action, War)
  Bruker 11928 (sim=0.679, topp-sjangre: Drama, Action, Adventure)
